# Heatmaps: delays, bunching and speed

All from the gold tables that store **sums per day × hour × line × stop** (`gold_delay_hour`,
`gold_headway_hour`, `gold_speed_hour`), so every filter below gives correct percentages and averages:
the sums are added up first and divided last.

- **Delays**: % on time and average delay (on time = -60 s ... < 300 s)
- **Bunching**: % of buses arriving bunched (< 0.25 × scheduled headway) and % large gaps (> 2 ×)
- **Speed**: average stop-to-stop speed (distance-weighted, incl. dwell at the stop)

Cells with fewer than `MIN_N` observations are left blank so thin data does not look like a signal.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

# the project folder = the first parent that contains analysis/ (works wherever the notebook sits)
REPO = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents] if (p / "analysis").is_dir())
GOLD = REPO / "tables" / "gold-layer"
MAP = REPO / "dashboard" / "map"
OUT = REPO / "tables" / "notebook-output"
OUT.mkdir(parents=True, exist_ok=True)

def gold(name, columns=None):
    return pd.read_parquet(GOLD / f"{name}.parquet", columns=columns)

meta = gold("gold_meta").set_index("key")["value"]
rules = gold("gold_thresholds").set_index("rule")["value"]
print("Gold service dates:", meta["service_dates"])
print("On time:", meta["on_time_rule"], "| bunching:", meta.get("bunching_rule", "?"))

In [ ]:
# ---- filters: change these and run the notebook again (None = everything) ----
WEEKS = None          # ISO weeks, e.g. [44]            (the data covers weeks 43-45)
DATES = None          # service dates, e.g. ["2026-10-28", "2026-10-29"]
WEEKDAYS = None       # 0 = Monday ... 6 = Sunday, e.g. [0, 1, 2, 3, 4] for working days
HOURS = None          # local hours, e.g. range(7, 9) for 07:00-08:59
LINES = None          # route numbers as text, e.g. ["3", "X60"]
MIN_N = 20            # hide cells / segments with fewer observations than this

def pick(df, hour_col="local_hour", line_col="route_short_name"):
    """Apply the filters above to any gold table that has these columns."""
    m = pd.Series(True, index=df.index)
    if WEEKS is not None and "iso_week" in df:      m &= df["iso_week"].isin(WEEKS)
    if DATES is not None and "service_date" in df:  m &= df["service_date"].astype(str).isin(DATES)
    if WEEKDAYS is not None and "weekday" in df:    m &= df["weekday"].isin(WEEKDAYS)
    if HOURS is not None and hour_col in df:        m &= df[hour_col].isin(list(HOURS))
    if LINES is not None and line_col in df:        m &= df[line_col].astype(str).isin([str(x) for x in LINES])
    return df[m]

TOP_LINES = 25        # lines shown in the line x hour heatmaps (busiest first)
TOP_STOPS = 25        # stops shown in the stop x hour heatmaps
WEEKDAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

In [ ]:
delay = pick(gold("gold_delay_hour", ["service_date", "iso_week", "weekday", "local_hour", "line_ref", "route_short_name",
                                      "stop_ref", "n_valid", "n_on_time", "n_late", "sum_delay_s"]))
headway = pick(gold("gold_headway_hour"))
speed = pick(gold("gold_speed_hour"))
stops = gold("dim_stop").set_index("stop_ref")["stop_name"]
delay = delay[delay.local_hour >= 0]
print(f"{delay.n_valid.sum():,} stop visits | {headway.n_headways.sum():,} headways | {speed.n_runs.sum():,} segment runs")

def heat(pivot, title, label, cmap, vmin=None, vmax=None, fmt="{:.0f}", figsize=None):
    """One heatmap; NaN cells stay blank."""
    figsize = figsize or (13, max(3, 0.32 * len(pivot) + 1.5))
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(pivot.values.astype(float), aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(pivot.shape[1])); ax.set_xticklabels(pivot.columns, fontsize=8)
    ax.set_yticks(range(pivot.shape[0])); ax.set_yticklabels(pivot.index, fontsize=8)
    if pivot.size <= 400:
        for (i, j), v in np.ndenumerate(pivot.values.astype(float)):
            if not np.isnan(v):
                ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=6.5)
    ax.set_title(title); plt.colorbar(im, ax=ax, label=label, shrink=.8); plt.tight_layout(); plt.show()

def ratio_pivot(df, index, num, den, min_n=None, scale=1.0, order=None):
    """sum(num) / sum(den) per (index, local_hour); cells with too few observations -> NaN."""
    g = df.groupby([index, "local_hour"])[[num, den]].sum()
    v = (g[num] / g[den] * scale).where(g[den] >= (min_n or MIN_N))
    p = v.unstack("local_hour")
    return p.reindex(order) if order is not None else p

def speed_pivot(df, index, order=None):
    g = df.groupby([index, "local_hour"])[["sum_distance_m", "sum_run_s", "n_runs"]].sum()
    v = (g.sum_distance_m / g.sum_run_s * 3.6).where(g.n_runs >= MIN_N)
    p = v.unstack("local_hour")
    return p.reindex(order) if order is not None else p

## 1. Weekday × hour

In [ ]:
for df, num, den, scale, title, label, cmap, lim in [
    (delay, "n_on_time", "n_valid", 100, "% on time", "%", "RdYlGn", (50, 95)),
    (delay, "sum_delay_s", "n_valid", 1, "Average delay (s)", "seconds", "RdYlGn_r", (0, 400)),
    (headway, "n_bunched", "n_headways", 100, "% bunched", "%", "Reds", (0, None)),
    (headway, "n_large_gap", "n_headways", 100, "% large gaps", "%", "Blues", (0, None))]:
    p = ratio_pivot(df, "weekday", num, den, scale=scale).rename(index=dict(enumerate(WEEKDAY_NAMES)))
    heat(p, f"{title}: weekday × hour", label, cmap, *lim, figsize=(13, 3.6))
p = speed_pivot(speed, "weekday").rename(index=dict(enumerate(WEEKDAY_NAMES)))
heat(p, "Average speed (km/h): weekday × hour", "km/h", "RdYlGn", 15, 40, figsize=(13, 3.6))

## 2. Line × hour (busiest lines)

In [ ]:
lines = delay.groupby("route_short_name").n_valid.sum().nlargest(TOP_LINES).index
d = delay[delay.route_short_name.isin(lines)]
heat(ratio_pivot(d, "route_short_name", "n_on_time", "n_valid", scale=100, order=lines),
     "% on time: line × hour", "%", "RdYlGn", 50, 95)
heat(ratio_pivot(d, "route_short_name", "sum_delay_s", "n_valid", order=lines),
     "Average delay (s): line × hour", "seconds", "RdYlGn_r", 0, 400)
h = headway[headway.route_short_name.isin(lines)]
heat(ratio_pivot(h, "route_short_name", "n_bunched", "n_headways", scale=100, order=lines),
     "% bunched: line × hour", "%", "Reds", 0, None)
s = speed[speed.route_short_name.isin(lines)]
heat(speed_pivot(s, "route_short_name", order=lines), "Average speed (km/h): line × hour", "km/h", "RdYlGn", 15, 40)

## 3. Stop × hour (busiest stops)

In [ ]:
top_stops = delay.groupby("stop_ref").n_valid.sum().nlargest(TOP_STOPS).index
names = [f"{stops.get(sr, sr)} ({sr.split(':')[-1]})" for sr in top_stops]   # stop name + id, names repeat per platform
d = delay[delay.stop_ref.isin(top_stops)]
heat(ratio_pivot(d, "stop_ref", "sum_delay_s", "n_valid", order=top_stops).set_axis(names),
     "Average delay (s): stop × hour", "seconds", "RdYlGn_r", 0, 400)
h = headway[headway.stop_ref.isin(top_stops)]
heat(ratio_pivot(h, "stop_ref", "n_bunched", "n_headways", order=top_stops, scale=100).set_axis(names),
     "% bunched: stop × hour", "%", "Reds", 0, None)

## 4. Date × hour (how days differ)

In [ ]:
heat(ratio_pivot(delay, "service_date", "n_on_time", "n_valid", scale=100), "% on time: date × hour", "%", "RdYlGn", 50, 95)
heat(ratio_pivot(headway, "service_date", "n_bunched", "n_headways", scale=100), "% bunched: date × hour", "%", "Reds", 0, None)
heat(speed_pivot(speed, "service_date"), "Average speed (km/h): date × hour", "km/h", "RdYlGn", 15, 40)